# Care Home & Inmate Welfare Dashboard
**Name:** Noor Arfa  
**Reg. No.:** 25SSDS415045  
**Tools:** Python, Tkinter, SQLite3

Tables: `institutions` → `inmate_records` → `care_provisions` (foreign keys with `ON DELETE CASCADE`).  
Run the cells from top to bottom. The last cell opens the window.

## 1. Imports and settings

In [ ]:
import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox
from datetime import date

DB_FILE = "care_home.db"

## 2. Database helpers
connect() turns on foreign keys. run_select() reads data and run_change() saves changes.

In [ ]:
def connect():
    conn = sqlite3.connect(DB_FILE)
    conn.execute("PRAGMA foreign_keys = ON")   # needed, otherwise cascade does not work
    return conn


def run_select(sql, params=()):
    conn = connect()
    try:
        cur = conn.cursor()
        cur.execute(sql, params)
        rows = cur.fetchall()
        columns = [c[0] for c in cur.description]
        return columns, rows
    finally:
        conn.close()            # always close, even if there is an error


def run_change(sql, params=()):
    conn = connect()
    try:
        cur = conn.cursor()
        cur.execute(sql, params)
        conn.commit()
    finally:
        conn.close()

## 3. Create the three tables (foreign keys + cascade delete)

In [ ]:
def create_tables():
    conn = connect()
    cur = conn.cursor()
    cur.execute("""CREATE TABLE IF NOT EXISTS institutions (
        institution_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL UNIQUE COLLATE NOCASE,
        contact_email TEXT NOT NULL)""")
    cur.execute("""CREATE TABLE IF NOT EXISTS inmate_records (
        record_id INTEGER PRIMARY KEY AUTOINCREMENT,
        institution_id INTEGER NOT NULL,
        inmate_name TEXT NOT NULL,
        admission_date TEXT NOT NULL,
        FOREIGN KEY (institution_id) REFERENCES institutions (institution_id)
            ON DELETE CASCADE)""")
    cur.execute("""CREATE TABLE IF NOT EXISTS care_provisions (
        provision_id INTEGER PRIMARY KEY AUTOINCREMENT,
        record_id INTEGER NOT NULL,
        provision_type TEXT NOT NULL,
        provision_amount REAL NOT NULL,
        FOREIGN KEY (record_id) REFERENCES inmate_records (record_id)
            ON DELETE CASCADE)""")
    conn.commit()
    conn.close()

## 4. Add sample data (only when the database is empty)

In [ ]:
def add_sample_data():
    # only add the sample rows when the database is empty
    cols, rows = run_select("SELECT COUNT(*) FROM institutions")
    if rows[0][0] > 0:
        return
    conn = connect()
    cur = conn.cursor()
    cur.execute("INSERT INTO institutions (name, contact_email) VALUES (?, ?)",
                ("Sri Sharada Old Age Home", "super.sharada@welfare.gov.in"))
    cur.execute("INSERT INTO institutions (name, contact_email) VALUES (?, ?)",
                ("Karunalaya Orphanage", "super.karunalaya@welfare.gov.in"))
    cur.execute("INSERT INTO institutions (name, contact_email) VALUES (?, ?)",
                ("Asha Destitute Shelter", "super.asha@welfare.gov.in"))

    # (institution_id, inmate name, admission date)
    inmates = [(1, "Kamala Bai", "2024-02-14"), (1, "Narayana Rao", "2024-06-03"),
               (2, "Master Arjun", "2023-11-21"), (2, "Baby Lakshmi", "2025-01-10"),
               (3, "Raju Shetty", "2025-03-28")]
    for inst_id, name, adm in inmates:
        cur.execute("INSERT INTO inmate_records (institution_id, inmate_name, admission_date) "
                    "VALUES (?, ?, ?)", (inst_id, name, adm))

    # (record_id, provision type, amount)
    provisions = [(1, "Food Ration", 1800), (1, "Medical Care", 2500),
                  (2, "Monthly Stipend", 1500), (3, "Clothing", 900),
                  (3, "Food Ration", 2100), (4, "Medical Care", 3200),
                  (5, "Food Ration", 1500), (5, "Clothing", 700)]
    for rec_id, ptype, amount in provisions:
        cur.execute("INSERT INTO care_provisions (record_id, provision_type, provision_amount) "
                    "VALUES (?, ?, ?)", (rec_id, ptype, amount))
    conn.commit()
    conn.close()

## 5. Check the amount and add a record

In [ ]:
def check_amount(text):
    # returns the amount as a number, or raises ValueError with a message
    try:
        amount = float(text)
    except ValueError:
        raise ValueError("Provision Amount must be a number.")
    if amount < 0:
        raise ValueError("Provision Amount cannot be negative.")
    if amount != amount or amount == float("inf"):
        raise ValueError("Provision Amount must be a normal number.")
    return amount


def add_record(inst_name, inmate_name, ptype, amount):
    conn = connect()
    try:
        cur = conn.cursor()
        # does the institution already exist?
        cur.execute("SELECT institution_id FROM institutions WHERE name = ?", (inst_name,))
        row = cur.fetchone()
        if row is None:
            email = "superintendent@" + inst_name.lower().replace(" ", "") + ".example"
            cur.execute("INSERT INTO institutions (name, contact_email) VALUES (?, ?)",
                        (inst_name, email))
            inst_id = cur.lastrowid
            is_new = True
        else:
            inst_id = row[0]
            is_new = False
        cur.execute("INSERT INTO inmate_records (institution_id, inmate_name, admission_date) "
                    "VALUES (?, ?, ?)", (inst_id, inmate_name, date.today().isoformat()))
        rec_id = cur.lastrowid
        cur.execute("INSERT INTO care_provisions (record_id, provision_type, provision_amount) "
                    "VALUES (?, ?, ?)", (rec_id, ptype, amount))
        conn.commit()
        return rec_id, is_new
    finally:
        conn.close()

## 6. Delete functions and the JOIN query

In [ ]:
# what can be deleted: name in dropdown -> (table, id column)
DELETE_OPTIONS = {
    "Care Institution": ("institutions", "institution_id"),
    "Inmate Record": ("inmate_records", "record_id"),
    "Care Provision": ("care_provisions", "provision_id"),
}


def delete_row(kind, row_id):
    table, id_column = DELETE_OPTIONS[kind]
    run_change("DELETE FROM " + table + " WHERE " + id_column + " = ?", (row_id,))


def get_delete_items(kind):
    # list of (text shown in dropdown, id)
    if kind == "Care Institution":
        cols, rows = run_select("SELECT institution_id, name FROM institutions")
        return [(str(r[0]) + " - " + r[1], r[0]) for r in rows]
    if kind == "Inmate Record":
        cols, rows = run_select("SELECT record_id, inmate_name FROM inmate_records")
        return [(str(r[0]) + " - " + r[1], r[0]) for r in rows]
    cols, rows = run_select("SELECT provision_id, provision_type, provision_amount "
                            "FROM care_provisions")
    return [(str(r[0]) + " - " + r[1] + " Rs." + str(r[2]), r[0]) for r in rows]


# LEFT JOIN is used so an inmate still shows even if all provisions are deleted
JOIN_SQL = """
    SELECT r.record_id AS "Record ID", i.name AS "Institution Name",
           r.inmate_name AS "Inmate Name", r.admission_date AS "Admission Date",
           p.provision_type AS "Provision Type", p.provision_amount AS "Provision Amount"
    FROM inmate_records r
    JOIN institutions i ON i.institution_id = r.institution_id
    LEFT JOIN care_provisions p ON p.record_id = r.record_id
    ORDER BY r.record_id, p.provision_id"""

## 7. Window functions (show tables, buttons)

In [ ]:
def show_table(tree, sql):
    columns, rows = run_select(sql)
    tree.delete(*tree.get_children())
    tree["columns"] = columns
    for col in columns:
        tree.heading(col, text=col)
        tree.column(col, width=130)
    for row in rows:
        tree.insert("", "end", values=row)


def refresh_tabs():
    show_table(tree_join, JOIN_SQL)
    show_table(tree_inst, "SELECT * FROM institutions")
    show_table(tree_inmates, "SELECT * FROM inmate_records")
    show_table(tree_prov, "SELECT * FROM care_provisions")


def refresh_delete_list(event=None):
    global delete_ids
    items = get_delete_items(kind_var.get())
    delete_ids = dict(items)
    item_box["values"] = [text for text, _ in items]
    item_var.set("")


def add_clicked():
    inst = inst_var.get().strip()
    inmate = inmate_var.get().strip()
    ptype = type_var.get().strip()
    amount_text = amount_var.get().strip()

    if inst == "" or inmate == "" or ptype == "" or amount_text == "":
        messagebox.showwarning("Missing data", "Please fill in all four fields.")
        return
    try:
        amount = check_amount(amount_text)
    except ValueError as error:
        messagebox.showerror("Invalid amount", str(error))
        return

    rec_id, is_new = add_record(inst, inmate, ptype, amount)
    if is_new:
        messagebox.showinfo("Saved", "Record " + str(rec_id) + " added (new institution created).")
    else:
        messagebox.showinfo("Saved", "Record " + str(rec_id) + " added (existing institution).")
    for var in (inst_var, inmate_var, type_var, amount_var):
        var.set("")
    refresh_tabs()
    refresh_delete_list()


def delete_clicked():
    choice = item_var.get()
    if choice == "":
        messagebox.showwarning("Nothing selected", "Please select an item to delete.")
        return
    if not messagebox.askyesno("Confirm", "Delete " + choice + " ?"):
        return
    delete_row(kind_var.get(), delete_ids[choice])
    refresh_tabs()
    refresh_delete_list()


def make_tab(notebook, title):
    frame = ttk.Frame(notebook)
    notebook.add(frame, text=title)
    tree = ttk.Treeview(frame, show="headings")
    scroll = ttk.Scrollbar(frame, orient="vertical", command=tree.yview)
    tree.configure(yscrollcommand=scroll.set)
    tree.pack(side="left", fill="both", expand=True)
    scroll.pack(side="right", fill="y")
    return tree

## 8. Build the window (Insert panel, Delete panel, 4 tabs)

In [ ]:
def open_window():
    global inst_var, inmate_var, type_var, amount_var, kind_var, item_var
    global item_box, delete_ids, tree_join, tree_inst, tree_inmates, tree_prov

    root = tk.Tk()
    root.title("Care Home & Inmate Welfare Dashboard")
    root.geometry("1000x650")

    inst_var = tk.StringVar()
    inmate_var = tk.StringVar()
    type_var = tk.StringVar()
    amount_var = tk.StringVar()
    kind_var = tk.StringVar(value="Care Institution")
    item_var = tk.StringVar()
    delete_ids = {}

    ttk.Label(root, text="Care Home & Inmate Welfare Dashboard",
              font=("Arial", 16, "bold")).pack(pady=8)

    top = ttk.Frame(root)
    top.pack(fill="x", padx=10)

    # insert panel
    insert_frame = ttk.LabelFrame(top, text="Insert Panel")
    insert_frame.pack(side="left", fill="both", expand=True, padx=5)
    labels = ["Institution Name", "Inmate Name", "Provision Type", "Provision Amount (Rs.)"]
    variables = [inst_var, inmate_var, type_var, amount_var]
    for i in range(4):
        ttk.Label(insert_frame, text=labels[i]).grid(row=i, column=0, padx=8, pady=4, sticky="w")
        if i == 2:
            box = ttk.Combobox(insert_frame, textvariable=variables[i], width=28,
                               values=["Food Ration", "Medical Care", "Clothing", "Monthly Stipend"])
        else:
            box = ttk.Entry(insert_frame, textvariable=variables[i], width=30)
        box.grid(row=i, column=1, padx=8, pady=4)
    ttk.Button(insert_frame, text="Add Record", command=add_clicked).grid(
        row=4, column=0, columnspan=2, pady=8)

    # delete panel
    delete_frame = ttk.LabelFrame(top, text="Delete Panel")
    delete_frame.pack(side="left", fill="both", expand=True, padx=5)
    ttk.Label(delete_frame, text="Delete a").grid(row=0, column=0, padx=8, pady=6, sticky="w")
    kind_box = ttk.Combobox(delete_frame, textvariable=kind_var, state="readonly",
                            values=list(DELETE_OPTIONS), width=32)
    kind_box.grid(row=0, column=1, padx=8, pady=6)
    kind_box.bind("<<ComboboxSelected>>", refresh_delete_list)
    ttk.Label(delete_frame, text="Select item").grid(row=1, column=0, padx=8, pady=6, sticky="w")
    item_box = ttk.Combobox(delete_frame, textvariable=item_var, state="readonly", width=32)
    item_box.grid(row=1, column=1, padx=8, pady=6)
    ttk.Button(delete_frame, text="Delete Selected", command=delete_clicked).grid(
        row=2, column=0, columnspan=2, pady=8)

    # four tabs
    notebook = ttk.Notebook(root)
    notebook.pack(fill="both", expand=True, padx=10, pady=10)
    tree_join = make_tab(notebook, "Combined View (JOIN)")
    tree_inst = make_tab(notebook, "Institutions")
    tree_inmates = make_tab(notebook, "Inmate Records")
    tree_prov = make_tab(notebook, "Care Provisions")

    refresh_tabs()
    refresh_delete_list()
    root.mainloop()

## 9. Run the program

In [ ]:
create_tables()
add_sample_data()
open_window()   # run this on your own computer (Tkinter needs a screen)